# Q1 Broad Non-MCQ Finite-Choice Extension

This Kaggle notebook extracts finite-choice reliability features for:

- **Models**: Qwen2.5-7B-Instruct, Llama-3.1-8B-Instruct, Mistral-7B-Instruct-v0.3
- **Non-MCQ tasks**: AG News-4, TREC-6, DBPedia-14
- Then combines these with the existing MCQ features and Banking77, runs locked validation, and creates final ZIP outputs.

## Required Kaggle Inputs

Add these datasets/files as Kaggle inputs before running:

1. Existing MCQ features dataset, usually visible as `/kaggle/input/.../features/features`
2. Banking77 features package
3. `locked_validation_fast_v5.py` validation script
4. Hugging Face token in Kaggle Secrets:
   - Name: `HF_TOKEN`
   - Accept access for `meta-llama/Llama-3.1-8B-Instruct` on Hugging Face


## Cell 1 — Install packages


In [ ]:
!pip -q install -U "bitsandbytes>=0.46.1" accelerate transformers datasets sentencepiece huggingface_hub scipy scikit-learn tqdm


## Cell 2 — Hugging Face login


In [ ]:
import os

try:
    from kaggle_secrets import UserSecretsClient
    from huggingface_hub import login

    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    login(token=HF_TOKEN)
    os.environ["HF_TOKEN"] = HF_TOKEN
    print("HF login done.")
except Exception as e:
    print("HF token not loaded:", e)
    os.environ["HF_TOKEN"] = ""


## Cell 3 — Extraction script for AG News, TREC, DBPedia


In [ ]:
%%writefile /kaggle/working/extract_nonmcq_finitechoice.py
import os, json, time, gc, re
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_ID = os.environ["MODEL_ID"]
MODEL_SHORT = os.environ["MODEL_SHORT"]
HF_TOKEN = os.environ.get("HF_TOKEN", None) or None

TASKS = [x.strip() for x in os.environ.get("TASKS", "agnews,trec6,dbpedia14").split(",") if x.strip()]
OUTPUT_DIR = os.environ.get("OUTPUT_DIR", "/kaggle/working/features_nonmcq_finitechoice")
USE_4BIT = os.environ.get("USE_4BIT", "1") == "1"
MAX_LENGTH = int(os.environ.get("MAX_LENGTH", "768"))
OPTION_BATCH_SIZE = int(os.environ.get("OPTION_BATCH_SIZE", "4"))

AGNEWS_PER_CLASS = int(os.environ.get("AGNEWS_PER_CLASS", "250"))     # 4 x 250 = 1000
TREC_PER_CLASS = int(os.environ.get("TREC_PER_CLASS", "80"))          # 6 x 80 = 480
DBPEDIA_PER_CLASS = int(os.environ.get("DBPEDIA_PER_CLASS", "100"))   # 14 x 100 = 1400

os.makedirs(OUTPUT_DIR, exist_ok=True)

TASK_CONFIGS = {
    "agnews": {
        "display_name": "AG News",
        "dataset_id": "ag_news",
        "split": "test",
        "text_fields": ["text"],
        "label_field": "label",
        "labels": ["World", "Sports", "Business", "Sci/Tech"],
        "per_class": AGNEWS_PER_CLASS,
    },
    "trec6": {
        "display_name": "TREC",
        "dataset_id": "trec",
        "split": "train",
        "text_fields": ["text"],
        "label_field": "coarse_label",
        "labels": ["Abbreviation", "Entity", "Description", "Human", "Location", "Numeric"],
        "per_class": TREC_PER_CLASS,
    },
    "dbpedia14": {
        "display_name": "DBPedia",
        "dataset_id": "dbpedia_14",
        "split": "test",
        "text_fields": ["title", "content"],
        "label_field": "label",
        "labels": [
            "Company",
            "Educational institution",
            "Artist",
            "Athlete",
            "Office holder",
            "Mean of transportation",
            "Building",
            "Natural place",
            "Village",
            "Animal",
            "Plant",
            "Album",
            "Film",
            "Written work",
        ],
        "per_class": DBPEDIA_PER_CLASS,
    },
}

LETTERS = list("ABCDEFGHIJKLMNOPQRSTUVWXYZ")

def safe_col_name(s):
    s = s.lower()
    s = re.sub(r"[^a-z0-9]+", "_", s)
    s = re.sub(r"_+", "_", s).strip("_")
    return s[:40]

def load_balanced_task(task_name):
    cfg = TASK_CONFIGS[task_name]
    ds = load_dataset(cfg["dataset_id"], split=cfg["split"])

    rows = []
    for ex in ds:
        parts = []
        for f in cfg["text_fields"]:
            if f in ex and ex[f] is not None:
                parts.append(str(ex[f]))
        text = ". ".join(parts).strip()
        label = int(ex[cfg["label_field"]])
        rows.append({"text": text, "label": label})

    df = pd.DataFrame(rows).dropna().reset_index(drop=True)
    n_classes = len(cfg["labels"])
    per_class = int(cfg["per_class"])

    sampled = []
    for y in range(n_classes):
        sub = df[df["label"] == y]
        if len(sub) < per_class:
            print(f"WARNING: {task_name} class {y} has only {len(sub)} examples; using all.")
            sampled.append(sub.sample(n=len(sub), random_state=42))
        else:
            sampled.append(sub.sample(n=per_class, random_state=42))

    out = (
        pd.concat(sampled, ignore_index=True)
          .sample(frac=1.0, random_state=42)
          .reset_index(drop=True)
    )

    print(f"\nLoaded {task_name}: {out.shape}")
    print("Class counts:")
    print(out["label"].value_counts().sort_index())

    return out, cfg

def make_prompt(tokenizer, text, labels):
    choice_lines = []
    for i, lab in enumerate(labels):
        choice_lines.append(f"{LETTERS[i]}. {lab}")

    user_content = (
        "Classify the following text into exactly one category.\n\n"
        f"Text:\n{text}\n\n"
        "Choices:\n"
        + "\n".join(choice_lines)
        + "\n\n"
        f"Return only the letter {LETTERS[0]} to {LETTERS[len(labels)-1]}."
    )

    messages = [{"role": "user", "content": user_content}]

    try:
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )
    except Exception:
        return user_content + "\nAnswer:"

@torch.no_grad()
def score_options(model, tokenizer, prompt, letters):
    scores = []

    for start in range(0, len(letters), OPTION_BATCH_SIZE):
        batch_letters = letters[start:start + OPTION_BATCH_SIZE]
        option_texts = [" " + x for x in batch_letters]
        texts = [prompt + x for x in option_texts]

        prompt_ids = tokenizer(
            prompt,
            add_special_tokens=True,
            truncation=True,
            max_length=MAX_LENGTH,
        )["input_ids"]

        prompt_len = len(prompt_ids)

        enc = tokenizer(
            texts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH,
            add_special_tokens=True,
        ).to(model.device)

        input_ids = enc["input_ids"]
        attn = enc["attention_mask"]

        labels = input_ids.clone()
        labels[:, :prompt_len] = -100
        labels[attn == 0] = -100

        out = model(input_ids=input_ids, attention_mask=attn)
        logits = out.logits

        shift_logits = logits[:, :-1, :]
        shift_labels = labels[:, 1:]
        mask = shift_labels != -100

        log_probs = torch.log_softmax(shift_logits, dim=-1)
        safe_labels = shift_labels.clone()
        safe_labels[~mask] = 0

        tok_logp = log_probs.gather(-1, safe_labels.unsqueeze(-1)).squeeze(-1)
        tok_logp = tok_logp * mask

        lengths = mask.sum(dim=1).clamp(min=1)
        mean_logp = tok_logp.sum(dim=1) / lengths

        scores.extend(mean_logp.detach().float().cpu().numpy().tolist())

        del enc, input_ids, attn, labels, out, logits
        torch.cuda.empty_cache()

    return np.asarray(scores, dtype=np.float32)

@torch.no_grad()
def get_hidden(model, tokenizer, prompt):
    enc = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_LENGTH,
        add_special_tokens=True,
    ).to(model.device)

    out = model(
        **enc,
        output_hidden_states=True,
        use_cache=False,
    )

    h = out.hidden_states[-1]
    last_idx = enc["attention_mask"].sum(dim=1).item() - 1
    vec = h[0, last_idx, :].detach().float().cpu().numpy()

    del enc, out, h
    torch.cuda.empty_cache()

    return vec

def softmax_np(scores):
    scores = np.asarray(scores, dtype=np.float64)
    z = scores - np.max(scores)
    p = np.exp(z)
    return p / p.sum()

def entropy_np(p):
    p = np.asarray(p, dtype=np.float64)
    return float(-np.sum(p * np.log(p + 1e-12)))

def load_model_and_tokenizer():
    tokenizer = AutoTokenizer.from_pretrained(
        MODEL_ID,
        token=HF_TOKEN,
        trust_remote_code=True,
    )

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model_kwargs = {
        "device_map": "auto",
        "trust_remote_code": True,
        "token": HF_TOKEN,
    }

    if USE_4BIT:
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type="nf4",
        )
        model_kwargs["quantization_config"] = bnb_config
    else:
        model_kwargs["torch_dtype"] = torch.float16

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        **model_kwargs,
    )

    model.eval()
    return tokenizer, model

def run_task(task_name, tokenizer, model):
    cfg = TASK_CONFIGS[task_name]
    labels = cfg["labels"]
    letters = LETTERS[:len(labels)]

    feat_path = os.path.join(OUTPUT_DIR, f"{MODEL_SHORT}__{task_name}__features.csv")
    hid_path = os.path.join(OUTPUT_DIR, f"{MODEL_SHORT}__{task_name}__hidden.npz")
    meta_path = os.path.join(OUTPUT_DIR, f"{MODEL_SHORT}__{task_name}__meta.json")

    if os.path.exists(feat_path) and os.path.exists(hid_path) and os.path.exists(meta_path):
        print(f"\nSKIP existing output: {MODEL_SHORT} {task_name}")
        return

    df, cfg = load_balanced_task(task_name)

    rows = []
    hidden_rows = []
    start_time = time.time()

    for i, r in tqdm(df.iterrows(), total=len(df), desc=f"{MODEL_SHORT} {task_name}"):
        text = str(r["text"])
        gold = int(r["label"])
        prompt = make_prompt(tokenizer, text, labels)

        scores = score_options(model, tokenizer, prompt, letters)
        probs = softmax_np(scores)

        pred = int(np.argmax(scores))
        correct = int(pred == gold)

        sorted_scores = np.sort(scores)[::-1]
        margin = float(sorted_scores[0] - sorted_scores[1])
        ent = entropy_np(probs)

        h = get_hidden(model, tokenizer, prompt)
        hidden_rows.append(h)

        rec = {
            "example_id": i,
            "model": MODEL_SHORT,
            "dataset": task_name,
            "gold_label": gold,
            "pred_label": pred,
            "correct": correct,

            "max_option_score": float(np.max(scores)),
            "mean_option_score": float(np.mean(scores)),
            "std_option_score": float(np.std(scores)),
            "min_option_score": float(np.min(scores)),
            "option_score_margin": margin,
            "option_score_entropy": ent,
            "top_prob_softmax": float(np.max(probs)),
            "n_options": int(len(labels)),
        }

        for j, lab in enumerate(labels):
            suffix = safe_col_name(f"{letters[j]}_{lab}")
            rec[f"score_{suffix}"] = float(scores[j])
            rec[f"prob_{suffix}"] = float(probs[j])

        rows.append(rec)

    feat = pd.DataFrame(rows)
    H = np.vstack(hidden_rows).astype("float32")

    feat.to_csv(feat_path, index=False)
    np.savez_compressed(hid_path, hidden=H)

    meta = {
        "model_id": MODEL_ID,
        "model_short": MODEL_SHORT,
        "dataset": task_name,
        "display_name": cfg["display_name"],
        "n": int(len(feat)),
        "accuracy": float(feat["correct"].mean()),
        "hidden_shape": list(H.shape),
        "labels": labels,
        "choice_letters": letters,
        "balanced_sampling": True,
        "per_class": int(cfg["per_class"]),
        "runtime_seconds": float(time.time() - start_time),
    }

    with open(meta_path, "w") as f:
        json.dump(meta, f, indent=2)

    print("\nDONE:", MODEL_SHORT, task_name)
    print("Accuracy:", meta["accuracy"])
    print("Hidden shape:", H.shape)
    print("Feature:", feat_path)
    print("Hidden:", hid_path)
    print("Meta:", meta_path)

def main():
    print("=" * 80)
    print("MODEL_ID:", MODEL_ID)
    print("MODEL_SHORT:", MODEL_SHORT)
    print("TASKS:", TASKS)
    print("OUTPUT_DIR:", OUTPUT_DIR)
    print("MAX_LENGTH:", MAX_LENGTH)
    print("OPTION_BATCH_SIZE:", OPTION_BATCH_SIZE)
    print("USE_4BIT:", USE_4BIT)
    print("=" * 80)

    tokenizer, model = load_model_and_tokenizer()

    for task in TASKS:
        if task not in TASK_CONFIGS:
            raise ValueError(f"Unknown task: {task}")
        run_task(task, tokenizer, model)
        gc.collect()
        torch.cuda.empty_cache()

    del model
    gc.collect()
    torch.cuda.empty_cache()

if __name__ == "__main__":
    main()


## Cell 4 — Run Qwen on all 3 non-MCQ datasets


In [ ]:
import os
from kaggle_secrets import UserSecretsClient

try:
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    HF_TOKEN = ""

os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["MODEL_ID"] = "Qwen/Qwen2.5-7B-Instruct"
os.environ["MODEL_SHORT"] = "qwen25_7b"
os.environ["TASKS"] = "agnews,trec6,dbpedia14"
os.environ["OUTPUT_DIR"] = "/kaggle/working/features_nonmcq_finitechoice"
os.environ["USE_4BIT"] = "1"
os.environ["MAX_LENGTH"] = "768"
os.environ["OPTION_BATCH_SIZE"] = "4"

os.environ["AGNEWS_PER_CLASS"] = "250"
os.environ["TREC_PER_CLASS"] = "80"
os.environ["DBPEDIA_PER_CLASS"] = "100"

!python /kaggle/working/extract_nonmcq_finitechoice.py


## Cell 5 — Clear GPU memory


In [ ]:
import gc, torch
gc.collect()
torch.cuda.empty_cache()
print("GPU memory cleared.")


## Cell 6 — Run Llama on all 3 non-MCQ datasets


In [ ]:
import os
from kaggle_secrets import UserSecretsClient

HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")

os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["MODEL_ID"] = "meta-llama/Llama-3.1-8B-Instruct"
os.environ["MODEL_SHORT"] = "llama31_8b_it"
os.environ["TASKS"] = "agnews,trec6,dbpedia14"
os.environ["OUTPUT_DIR"] = "/kaggle/working/features_nonmcq_finitechoice"
os.environ["USE_4BIT"] = "1"
os.environ["MAX_LENGTH"] = "768"
os.environ["OPTION_BATCH_SIZE"] = "4"

os.environ["AGNEWS_PER_CLASS"] = "250"
os.environ["TREC_PER_CLASS"] = "80"
os.environ["DBPEDIA_PER_CLASS"] = "100"

!python /kaggle/working/extract_nonmcq_finitechoice.py


## Cell 7 — Clear GPU memory


In [ ]:
import gc, torch
gc.collect()
torch.cuda.empty_cache()
print("GPU memory cleared.")


## Cell 8 — Run Mistral on all 3 non-MCQ datasets


In [ ]:
import os
from kaggle_secrets import UserSecretsClient

try:
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    HF_TOKEN = ""

os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["MODEL_ID"] = "mistralai/Mistral-7B-Instruct-v0.3"
os.environ["MODEL_SHORT"] = "mistral7b_v03"
os.environ["TASKS"] = "agnews,trec6,dbpedia14"
os.environ["OUTPUT_DIR"] = "/kaggle/working/features_nonmcq_finitechoice"
os.environ["USE_4BIT"] = "1"
os.environ["MAX_LENGTH"] = "768"
os.environ["OPTION_BATCH_SIZE"] = "4"

os.environ["AGNEWS_PER_CLASS"] = "250"
os.environ["TREC_PER_CLASS"] = "80"
os.environ["DBPEDIA_PER_CLASS"] = "100"

!python /kaggle/working/extract_nonmcq_finitechoice.py


## Cell 9 — Check extraction outputs


In [ ]:
import os, glob, json
import pandas as pd

ROOT = "/kaggle/working/features_nonmcq_finitechoice"

print("Files:")
for p in sorted(glob.glob(ROOT + "/*")):
    print(os.path.basename(p), round(os.path.getsize(p) / 1024 / 1024, 2), "MB")

print("\nMeta summary:")
rows = []
for p in sorted(glob.glob(ROOT + "/*meta.json")):
    with open(p) as f:
        meta = json.load(f)
    rows.append({
        "file": os.path.basename(p),
        "model": meta["model_short"],
        "dataset": meta["dataset"],
        "n": meta["n"],
        "accuracy": meta["accuracy"],
        "hidden_shape": meta["hidden_shape"],
        "runtime_min": meta["runtime_seconds"] / 60,
    })

df = pd.DataFrame(rows)
display(df)

print("\nExpected 9 meta rows:")
print(df.groupby("dataset")["model"].nunique())


## Cell 10 — Checkpoint ZIP for extracted non-MCQ features


In [ ]:
import os, glob, zipfile

SRC = "/kaggle/working/features_nonmcq_finitechoice"
ZIP = "/kaggle/working/features_nonmcq_finitechoice_checkpoint.zip"

files = glob.glob(SRC + "/*")

print("Files:", len(files))
for p in sorted(files):
    print(os.path.basename(p), round(os.path.getsize(p) / 1024 / 1024, 2), "MB")

assert len(files) >= 27, "Expected at least 27 files: 9 csv + 9 npz + 9 json."

with zipfile.ZipFile(ZIP, "w", zipfile.ZIP_DEFLATED) as z:
    for p in files:
        z.write(p, arcname=os.path.basename(p))

print("Created:", ZIP)
print("ZIP size MB:", os.path.getsize(ZIP) / 1024 / 1024)


## Cell 11 — Combine MCQ + Banking77 + non-MCQ features


In [ ]:
import os, glob, shutil
from collections import Counter

COMBINED = "/kaggle/working/features_combined_q1_broad_nonmcq"
os.makedirs(COMBINED, exist_ok=True)

for p in glob.glob(COMBINED + "/*"):
    os.remove(p)

# 1. Main MCQ features
mcq_csvs = [
    p for p in glob.glob("/kaggle/input/**/*features.csv", recursive=True)
    if "banking77" not in os.path.basename(p).lower()
    and "agnews" not in os.path.basename(p).lower()
    and "trec6" not in os.path.basename(p).lower()
    and "dbpedia14" not in os.path.basename(p).lower()
    and "partial" not in os.path.basename(p).lower()
]

assert len(mcq_csvs) > 0, "No MCQ feature files found."

folder_counts = Counter(os.path.dirname(p) for p in mcq_csvs)
MAIN = folder_counts.most_common(1)[0][0]
print("Detected MCQ folder:", MAIN)

for p in glob.glob(MAIN + "/*"):
    if os.path.isfile(p) and p.endswith((".csv", ".npz", ".json")):
        shutil.copy2(p, COMBINED)

# 2. Banking77
banking_files = [
    p for p in glob.glob("/kaggle/input/**/*banking77*", recursive=True)
    if os.path.isfile(p)
    and p.endswith((".csv", ".npz", ".json"))
    and "partial" not in os.path.basename(p).lower()
]

for p in banking_files:
    shutil.copy2(p, COMBINED)

# 3. Non-MCQ new features
nonmcq_files = [
    p for p in glob.glob("/kaggle/working/features_nonmcq_finitechoice/*", recursive=True)
    if os.path.isfile(p)
    and p.endswith((".csv", ".npz", ".json"))
    and "partial" not in os.path.basename(p).lower()
]

for p in nonmcq_files:
    shutil.copy2(p, COMBINED)

print("Combined folder:", COMBINED)
print("Total files:", len(glob.glob(COMBINED + "/*")))
print("CSV:", len(glob.glob(COMBINED + "/*.csv")))
print("NPZ:", len(glob.glob(COMBINED + "/*.npz")))
print("JSON:", len(glob.glob(COMBINED + "/*.json")))

print("\nExtension files:")
for p in sorted(
    glob.glob(COMBINED + "/*banking77*")
    + glob.glob(COMBINED + "/*agnews*")
    + glob.glob(COMBINED + "/*trec6*")
    + glob.glob(COMBINED + "/*dbpedia14*")
):
    print(os.path.basename(p))


## Cell 12 — Copy and patch locked validation script


In [ ]:
import os, glob, shutil, re

matches = (
    glob.glob("/kaggle/input/**/*locked_validation_fast_v5*.py", recursive=True)
    + glob.glob("/kaggle/working/**/*locked_validation_fast_v5*.py", recursive=True)
)

print("Found validation scripts:")
for m in matches:
    print(m)

assert len(matches) > 0, "No locked_validation_fast_v5 script found."

dst = "/kaggle/working/locked_validation_fast_v5.py"

if os.path.exists(dst):
    os.remove(dst)

shutil.copy2(matches[0], dst)

with open(dst, "r", encoding="utf-8") as f:
    s = f.read()

new_datasets = ["agnews", "trec6", "dbpedia14"]

m = re.search(r"(KNOWN_DATASETS\s*=\s*\[)(.*?)(\])", s, flags=re.S)

if m:
    prefix, block, suffix = m.group(1), m.group(2), m.group(3)
    for d in new_datasets:
        if d not in block:
            block = block.rstrip()
            if block.endswith(","):
                block += f'\n    "{d}",'
            else:
                block += f',\n    "{d}",'
    s = s[:m.start()] + prefix + block + "\n" + suffix + s[m.end():]
else:
    for d in new_datasets:
        if d not in s:
            s = s.replace('"banking77"', f'"banking77", "{d}"')
            s = s.replace("'banking77'", f"'banking77', '{d}'")

with open(dst, "w", encoding="utf-8") as f:
    f.write(s)

with open(dst, "r", encoding="utf-8") as f:
    check = f.read()

for d in new_datasets:
    print(d, "supported:", d in check)
    assert d in check, f"{d} was not patched into validation script."

print("Validation script ready:", dst)


## Cell 13 — Run locked validation


In [ ]:
import os

os.environ["SLM_FEATURE_INPUT"] = "/kaggle/working/features_combined_q1_broad_nonmcq"
os.environ["SLM_LOCKED_OUTPUT"] = "/kaggle/working/slm_locked_validation_outputs_q1_broad_nonmcq"
os.environ["SLM_BOOTSTRAP_B"] = "300"

print("Input:", os.environ["SLM_FEATURE_INPUT"])
print("Output:", os.environ["SLM_LOCKED_OUTPUT"])

!python /kaggle/working/locked_validation_fast_v5.py


## Cell 14 — Add hierarchical bootstrap, sensitivity, and cost tables


In [ ]:
import os, glob
import numpy as np
import pandas as pd

OUT = "/kaggle/working/slm_locked_validation_outputs_q1_broad_nonmcq"
FEATURE_DIR = "/kaggle/working/features_combined_q1_broad_nonmcq"

selected_path = os.path.join(OUT, "locked_primary_selected.csv")
discover_path = os.path.join(OUT, "discovered_conditions.csv")

assert os.path.exists(selected_path), "locked_primary_selected.csv not found."
assert os.path.exists(discover_path), "discovered_conditions.csv not found."

df = pd.read_csv(selected_path)
disc = pd.read_csv(discover_path)

print("Discovered datasets:")
print(disc["dataset"].value_counts())

required = ["agnews", "trec6", "dbpedia14", "banking77"]
for d in required:
    assert d in set(disc["dataset"]), f"Missing dataset in discovered conditions: {d}"

METRICS = [
    "test_auroc_correct",
    "test_auprc_failure",
    "test_risk_at_80",
    "test_aurc",
]

COMPARISONS = [
    "hidden_pca",
    "cheap_plus_hidden_pca",
]

def nested_condition_bootstrap(frame, name, B=3000, seed=123):
    rng = np.random.default_rng(seed)
    rows = []

    frame = frame.copy()
    cond_df = frame[["model", "dataset"]].drop_duplicates().reset_index(drop=True)
    conds = cond_df.values.tolist()

    for fam in COMPARISONS:
        for metric in METRICS:
            sub = frame[frame["family"].isin(["confidence_option", fam])].copy()

            piv = sub.pivot_table(
                index=["model", "dataset", "seed"],
                columns="family",
                values=metric,
                aggfunc="first",
            ).dropna()

            if piv.empty or fam not in piv.columns or "confidence_option" not in piv.columns:
                continue

            boots = []

            for _ in range(B):
                sampled_cond_idx = rng.choice(len(conds), size=len(conds), replace=True)
                vals = []

                for ci in sampled_cond_idx:
                    model, dataset = conds[ci]

                    mask = (
                        (piv.index.get_level_values("model") == model)
                        & (piv.index.get_level_values("dataset") == dataset)
                    )
                    cond_rows = piv.loc[mask]

                    if len(cond_rows) == 0:
                        continue

                    sampled_rows = cond_rows.iloc[rng.integers(0, len(cond_rows), len(cond_rows))]
                    delta = sampled_rows[fam].values - sampled_rows["confidence_option"].values
                    vals.extend(delta.tolist())

                if len(vals) > 0:
                    boots.append(float(np.mean(vals)))

            raw_delta = piv[fam] - piv["confidence_option"]

            rows.append({
                "analysis": name,
                "comparison": f"{fam} - confidence_option",
                "metric": metric,
                "n_condition_seed_pairs": int(len(raw_delta)),
                "n_conditions": int(piv.reset_index()[["model", "dataset"]].drop_duplicates().shape[0]),
                "mean_delta": float(raw_delta.mean()),
                "median_delta": float(raw_delta.median()),
                "ci_low": float(np.percentile(boots, 2.5)),
                "ci_high": float(np.percentile(boots, 97.5)),
                "positive_cases": int((raw_delta > 0).sum()),
                "negative_cases": int((raw_delta < 0).sum()),
            })

    return pd.DataFrame(rows)

main_mcq = df[~df["dataset"].isin(["banking77", "agnews", "trec6", "dbpedia14"])].copy()
nonmcq = df[df["dataset"].isin(["agnews", "trec6", "dbpedia14"])].copy()
extensions = df[df["dataset"].isin(["banking77", "agnews", "trec6", "dbpedia14"])].copy()
all_primary = df.copy()

boot_final = pd.concat([
    nested_condition_bootstrap(main_mcq, "main_mcq_24_conditions"),
    nested_condition_bootstrap(nonmcq, "non_mcq_9_conditions"),
    nested_condition_bootstrap(extensions, "all_extensions_10_conditions"),
    nested_condition_bootstrap(all_primary, "all_primary_conditions"),
], ignore_index=True)

boot_final.to_csv(os.path.join(OUT, "hierarchical_condition_bootstrap_deltas.csv"), index=False)

ext_table = extensions.groupby(["model", "dataset", "family"], as_index=False)[METRICS].mean()
ext_table.to_csv(os.path.join(OUT, "extension_results_by_model_dataset_family.csv"), index=False)

# Sensitivity excluding low-failure conditions
fail_path = os.path.join(OUT, "failure_counts_by_condition_seed.csv")
sens_rows = []

if os.path.exists(fail_path):
    fail = pd.read_csv(fail_path)
    print("\nFailure-count columns:", fail.columns.tolist())

    candidate_fail_cols = [c for c in fail.columns if "fail" in c.lower()]
    print("Candidate fail columns:", candidate_fail_cols)

    fail_col = None
    for c in ["n_failures", "test_failures", "failures", "n_test_failures"]:
        if c in fail.columns:
            fail_col = c
            break

    if fail_col is None and candidate_fail_cols:
        fail_col = candidate_fail_cols[0]

    if fail_col is not None:
        cond_fail = fail.groupby(["model", "dataset"], as_index=False)[fail_col].mean()
        for threshold in [10, 25, 50]:
            keep = cond_fail[cond_fail[fail_col] >= threshold][["model", "dataset"]]
            tmp = df.merge(keep, on=["model", "dataset"], how="inner")

            for fam in COMPARISONS:
                for metric in METRICS:
                    piv = tmp[tmp["family"].isin(["confidence_option", fam])].pivot_table(
                        index=["model", "dataset", "seed"],
                        columns="family",
                        values=metric,
                        aggfunc="first",
                    ).dropna()

                    if len(piv) == 0 or fam not in piv.columns:
                        continue

                    delta = piv[fam] - piv["confidence_option"]

                    sens_rows.append({
                        "threshold_mean_failures": threshold,
                        "comparison": f"{fam} - confidence_option",
                        "metric": metric,
                        "n_condition_seed_pairs": int(len(delta)),
                        "n_conditions": int(piv.reset_index()[["model", "dataset"]].drop_duplicates().shape[0]),
                        "mean_delta": float(delta.mean()),
                        "median_delta": float(delta.median()),
                        "positive_cases": int((delta > 0).sum()),
                        "negative_cases": int((delta < 0).sum()),
                    })

sens_df = pd.DataFrame(sens_rows)
sens_df.to_csv(os.path.join(OUT, "low_failure_sensitivity_summary.csv"), index=False)

# Hidden storage / cost table
cost_rows = []

for feat_path in glob.glob(FEATURE_DIR + "/*features.csv"):
    base = os.path.basename(feat_path)
    if "__features.csv" not in base:
        continue

    prefix = base.replace("__features.csv", "")
    hidden_path = os.path.join(FEATURE_DIR, prefix + "__hidden.npz")

    parts = prefix.split("__")
    model = "__".join(parts[:-1])
    dataset = parts[-1]

    feat_size = os.path.getsize(feat_path) if os.path.exists(feat_path) else 0
    hidden_size = os.path.getsize(hidden_path) if os.path.exists(hidden_path) else 0

    n, d = np.nan, np.nan
    hidden_key = ""

    if os.path.exists(hidden_path):
        z = np.load(hidden_path)
        for k in z.files:
            arr = z[k]
            if arr.ndim == 2:
                n, d = arr.shape
                hidden_key = k
                break

    try:
        csv_n = len(pd.read_csv(feat_path))
    except Exception:
        csv_n = np.nan

    cost_rows.append({
        "model": model,
        "dataset": dataset,
        "feature_csv_mb": feat_size / 1024 / 1024,
        "hidden_npz_mb": hidden_size / 1024 / 1024,
        "n_examples": csv_n,
        "hidden_dim": d,
        "hidden_key": hidden_key,
        "raw_hidden_float32_mb": (float(n) * float(d) * 4 / 1024 / 1024) if np.isfinite(n) and np.isfinite(d) else np.nan,
        "raw_hidden_float16_mb": (float(n) * float(d) * 2 / 1024 / 1024) if np.isfinite(n) and np.isfinite(d) else np.nan,
        "locked_probe_fits_per_condition": 105,
        "cheap_only_fits_per_condition": 15,
        "extra_hidden_probe_fits_per_condition": 90,
    })

cost_df = pd.DataFrame(cost_rows)
cost_df.to_csv(os.path.join(OUT, "hidden_probe_storage_and_training_costs.csv"), index=False)

print("\nSaved:")
print(" - hierarchical_condition_bootstrap_deltas.csv")
print(" - extension_results_by_model_dataset_family.csv")
print(" - low_failure_sensitivity_summary.csv")
print(" - hidden_probe_storage_and_training_costs.csv")

print("\nExtension table:")
display(ext_table)

print("\nHierarchical bootstrap:")
display(boot_final)

print("\nSensitivity:")
display(sens_df.head())

print("\nCost table:")
display(cost_df.head())


## Cell 15 — Zip final validation output


In [ ]:
import os, glob, zipfile

OUT = "/kaggle/working/slm_locked_validation_outputs_q1_broad_nonmcq"
ZIP = "/kaggle/working/slm_locked_validation_outputs_q1_broad_nonmcq.zip"

files = glob.glob(OUT + "/*")

print("Output files found:", len(files))
for p in sorted(files):
    print(os.path.basename(p), os.path.getsize(p), "bytes")

assert len(files) > 0, "Output folder is empty."

with zipfile.ZipFile(ZIP, "w", zipfile.ZIP_DEFLATED) as z:
    for p in files:
        z.write(p, arcname=os.path.basename(p))

print("Created:", ZIP)
print("ZIP size MB:", os.path.getsize(ZIP) / 1024 / 1024)
